In [1]:
import pandas as pd

# Load RNA-RNA interaction file
file_path = "RNAInter_RNA_RNA.txt"

# Try common delimiters (RNAInter files are usually tab-delimited)
df = pd.read_csv(file_path, sep='\t', low_memory=False)

# Display basic info
print(f"Number of interactions: {len(df)}")
df.head()


Number of interactions: 9484609


,RNAInterID,Interactor1.Symbol,Category1,Species1,Interactor2.Symbol,Category2,Species2,Raw_ID1,Raw_ID2,score,strong,weak,predict
0,RR00000169,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr4(CA)n125049227,chr15B1_Mus136600585,0.23,NaN,MARIO,NaN
1,RR00000238,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr8(CA)n67899847,chr8B1_Mus167899573,0.23,NaN,MARIO,NaN
2,RR00000198,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr6(CA)n135011645,chr6B1_Mus1135012241,0.23,NaN,MARIO,NaN
3,RR00000231,(Ca)N,repeats,Mus musculus,B1_Mus1,repeats,Mus musculus,chr8(CA)n114753246,chr8B1_Mus1114753067,0.23,NaN,MARIO,NaN
4,RR00000249,(Ca)N,repeats,Mus musculus,B3,repeats,Mus musculus,chr9(CA)n21755504,chr9B321755686,0.23,NaN,MARIO,NaN


In [2]:
# Show column names
print("Columns:", df.columns.tolist())

# Inspect data types
df.dtypes


Columns: ['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1', 'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2', 'score', 'strong', 'weak', 'predict']


RNAInterID             object
Interactor1.Symbol     object
Category1              object
Species1               object
Interactor2.Symbol     object
Category2              object
Species2               object
Raw_ID1                object
Raw_ID2                object
score                 float64
strong                 object
weak                   object
predict                object
dtype: object

In [3]:
# Check unique categories in both interactors
cat1_unique = df['Category1'].unique()
cat2_unique = df['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)



All unique RNA categories in dataset: {'snRNA', 'unknown', 'others', 'pseudo', 'protein', 'circRNA', 'miRNA', 'scRNA', 'Category2', 'sncRNA', 'processed_transcript', 'TR_C_gene', 'antisense', 'lincRNA', 'sRNA', 'unassigned RNA', 'ncRNA', 'PCG', 'rRNA', 'shRNA', 'repeats', 'Protein', 'TEC', 'snoRNA', 'piRNA', 'Category1', 'lncRNA', 'ribozyme', 'miscRNA', 'tRF', 'eRNA', 'scaRNA', 'mRNA', 'tRNA', 'TF'}


In [4]:
exclude_categories = {'Category1', 'Category2', 'unknown', 'unassigned RNA', 'others', 'Protein', 'protein',
                      'PCG', # Protein-coding Gene
                      'TF', # transcription factor gene
                      'TR_C_gene', # T-cell receptor constant region gene; immune-related.
                      'antisense', # transcripts complementary to sense strands of annotated genes.
                      'TEC', # “to be experimentally confirmed,” provisional transcripts.
                      'repeats' # transcripts overlapping repetitive genomic elements.
                     }

# Filter out unwanted categories from both interactors
df_filt = df[~df['Category1'].isin(exclude_categories) & ~df['Category2'].isin(exclude_categories)]

print(f"Rows after removing not-clear categories: {len(df_filt)}")


Rows after removing not-clear categories: 9429474


In [5]:
# Check unique categories in both interactors
cat1_unique = df_filt['Category1'].unique()
cat2_unique = df_filt['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)

All unique RNA categories in dataset: {'snRNA', 'pseudo', 'circRNA', 'miRNA', 'scRNA', 'sncRNA', 'processed_transcript', 'lincRNA', 'sRNA', 'ncRNA', 'rRNA', 'shRNA', 'snoRNA', 'piRNA', 'lncRNA', 'ribozyme', 'miscRNA', 'tRF', 'eRNA', 'scaRNA', 'mRNA', 'tRNA'}


In [6]:
# filter nan
print(f"RNA–RNA interactions: {len(df_filt)}")
df_filt = df_filt.dropna(subset=['Raw_ID1', 'Raw_ID2'])
print(f"RNA–RNA interactions: {len(df_filt)}")

RNA–RNA interactions: 9429474
RNA–RNA interactions: 8976390


In [7]:
parsed = []

def parse_ids(id_value):
    if pd.isna(id_value):
        return [None]
    elif ':' in id_value:
        prefix, ids_str = id_value.split(':', 1)
        return [f"{prefix}:{id_.strip()}" for id_ in ids_str.split(';')]
    else:
        parsed.append(id_value)
        return [id_value.strip()]

# Add list columns
df_filt['Raw_ID1_list'] = df_filt['Raw_ID1'].apply(parse_ids)
df_filt['Raw_ID2_list'] = df_filt['Raw_ID2'].apply(parse_ids)


In [8]:
# we need to standardise the species to TAXIDs
# 1. See unique species
df_filt.replace({'-': None}, inplace=True)
unique_species = pd.unique(df_filt[['Species1', 'Species2']].values.ravel())
print("Unique species:", len(unique_species))

from ete3 import NCBITaxa
ncbi = NCBITaxa()

unique_species_taxids = {}
missing = []

for test_species in unique_species:
    # Skip placeholders
    if test_species in [None]:
        continue

    # Handle known special cases first
    if test_species == 'Solanum tuberosum L.':
        unique_species_taxids[test_species] = "4113"
        continue
    if test_species == 'Rhodobacter sphaeroides ATCC 17025':
        unique_species_taxids[test_species] = "1061"
        continue

    # Query NCBI
    try:
        taxid = str(ncbi.get_name_translator([test_species])[test_species][0])
        unique_species_taxids[test_species] = taxid
    except KeyError:
        missing.append(test_species)

print("Mapped species (unique -1 for None):", len(unique_species_taxids))
print("Missing species:", missing)


Unique species: 98
Mapped species (unique -1 for None): 97
Missing species: []


In [9]:
# Assume df is your original dataframe
# unique_species_taxids: dict {species_name -> taxid}

df_filt['Species1_TaxID'] = df_filt['Species1'].map(unique_species_taxids)
df_filt['Species2_TaxID'] = df_filt['Species2'].map(unique_species_taxids)

unmapped_s1 = df_filt.loc[df_filt['Species1'].notna() & df_filt['Species1_TaxID'].isna(), 'Species1'].unique()
unmapped_s2 = df_filt.loc[df_filt['Species2'].notna() & df_filt['Species2_TaxID'].isna(), 'Species2'].unique()

print("Unmapped Species1:", unmapped_s1)
print("Unmapped Species2:", unmapped_s2)

Unmapped Species1: []
Unmapped Species2: []


In [10]:
print(f"RNA–RNA interactions: {len(df_filt)}")
# removing NAN
df_filt = df_filt.dropna(subset=['Species1_TaxID', 'Species2_TaxID'])
print(f"RNA–RNA interactions: {len(df_filt)}")

RNA–RNA interactions: 8976390
RNA–RNA interactions: 8976389


In [11]:
# Remove mRNA interactions at the 
print(f"RNA–RNA interactions: {len(df_filt)}")
df_filt = df_filt[
    (df_filt['Category1'] != 'mRNA') &
    (df_filt['Category2'] != 'mRNA')
].copy()
print(f"RNA–RNA interactions: {len(df_filt)}")

# Remove circRNA interactions at the beginning, 
df_filt = df_filt[
    (df_filt['Category1'] != 'circRNA') &
    (df_filt['Category2'] != 'circRNA')
].copy()
print(f"RNA–RNA interactions: {len(df_filt)}")

# Remove circRNA interactions at the beginning, 
df_filt = df_filt[
    (df_filt['Category1'] != 'tRF') &
    (df_filt['Category2'] != 'tRF')
].copy()
print(f"RNA–RNA interactions: {len(df_filt)}")

RNA–RNA interactions: 8976389
RNA–RNA interactions: 190245
RNA–RNA interactions: 189332
RNA–RNA interactions: 188828


In [12]:
# Check unique categories in both interactors
cat1_unique = df_filt['Category1'].unique()
cat2_unique = df_filt['Category2'].unique()

# Optionally, combined unique categories across both columns
combined_categories = set(cat1_unique) | set(cat2_unique)
print("All unique RNA categories in dataset:", combined_categories)

RNA_TYPE_MAP = {
    'processed_transcript': 'ncRNA',
    'miscRNA': 'ncRNA',
}

# Apply standardisation
df_filt['Category1_std'] = df_filt['Category1'].replace(RNA_TYPE_MAP)
df_filt['Category2_std'] = df_filt['Category2'].replace(RNA_TYPE_MAP)

# Recompute unique categories after mapping
cat1_std = set(df_filt['Category1_std'].unique())
cat2_std = set(df_filt['Category2_std'].unique())

print("Standardised categories:", cat1_std | cat2_std)

assert not df_filt[['Category1_std', 'Category2_std']].isna().any().any()

All unique RNA categories in dataset: {'snRNA', 'lincRNA', 'pseudo', 'sRNA', 'ncRNA', 'miRNA', 'rRNA', 'scRNA', 'snoRNA', 'piRNA', 'sncRNA', 'lncRNA', 'processed_transcript', 'ribozyme', 'miscRNA', 'scaRNA', 'tRNA'}
Standardised categories: {'snRNA', 'lincRNA', 'pseudo', 'sRNA', 'ncRNA', 'miRNA', 'rRNA', 'scRNA', 'snoRNA', 'piRNA', 'sncRNA', 'lncRNA', 'ribozyme', 'scaRNA', 'tRNA'}


In [13]:
# DATASET IS READY.
# df_filt keys: Index(['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1',
#       'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2',
#       'score', 'strong', 'weak', 'predict', 'Raw_ID1_list', 'Raw_ID2_list'],
#      dtype='object')

'''
Index(['RNAInterID', 'Interactor1.Symbol', 'Category1', 'Species1',
       'Interactor2.Symbol', 'Category2', 'Species2', 'Raw_ID1', 'Raw_ID2',
       'score', 'strong', 'weak', 'predict', 'Raw_ID1_list', 'Raw_ID2_list',
       'Species1_TaxID', 'Species2_TaxID'],
      dtype='object')
'''

# Now we need to extract sequences, we start by taking unique of ('Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_list')
# We do not care about 1 and 2, we can create the final database later.

def normalize_id_list(x):
    if isinstance(x, list):
        return tuple(sorted(str(i) for i in x if i is not None))
    return tuple()
df_filtbackup = df_filt

df_filt = df_filt.copy()
df_filt['Raw_ID1_key'] = df_filt['Raw_ID1_list'].apply(normalize_id_list)
df_filt['Raw_ID2_key'] = df_filt['Raw_ID2_list'].apply(normalize_id_list)

rna_entities_1 = (
    df_filt.loc[:, ['Interactor1.Symbol', 'Category1', 'Species1_TaxID', 'Raw_ID1_key']]
    .drop_duplicates()
    .rename(columns={
        'Interactor1.Symbol': 'Symbol',
        'Category1': 'Category',
        'Species1_TaxID': 'Species',
        'Raw_ID1_key': 'Raw_ID_key'
    })
)

rna_entities_2 = (
    df_filt.loc[:, ['Interactor2.Symbol', 'Category2', 'Species2_TaxID', 'Raw_ID2_key']]
    .drop_duplicates()
    .rename(columns={
        'Interactor2.Symbol': 'Symbol',
        'Category2': 'Category',
        'Species2_TaxID': 'Species',
        'Raw_ID2_key': 'Raw_ID_key'
    })
)

rna_entities = (
    pd.concat([rna_entities_1, rna_entities_2], ignore_index=True)
    .drop_duplicates()
)

print(f"Unique RNA entities: {len(rna_entities)}")

rna_entities['RNA_IDs'] = rna_entities['Raw_ID_key'].apply(list)

def classify_ids(id_list):
    return {
        'miRBase': [],
        'NCBI': [],
        'Ensembl': [],
        'circBase': [],
        'tRFdb': [],
        'GenBank': [],
        'Other': [],
        'RNAcentral': []   # ← to be filled after resolution
    }

def populate_buckets(id_list):
    buckets = classify_ids(id_list)
    for rid in id_list:
        if rid.startswith('miRBase:'):
            buckets['miRBase'].append(rid.split(':', 1)[1])
        elif rid.startswith('NCBI:'):
            buckets['NCBI'].append(rid.split(':', 1)[1])
        elif rid.startswith('Ensembl:'):
            buckets['Ensembl'].append(rid.split(':', 1)[1])
        elif rid.startswith('circBase:'):
            buckets['circBase'].append(rid.split(':', 1)[1])
        elif rid.startswith('tRFdb:'):
            buckets['tRFdb'].append(rid.split(':', 1)[1])
        elif rid.startswith('GenBank:'):
            buckets['GenBank'].append(rid.split(':', 1)[1])
        else:
            buckets['Other'].append(rid)
    return buckets

rna_entities['ID_buckets'] = rna_entities['RNA_IDs'].apply(populate_buckets)

import pandas as pd
df_filt.to_pickle("interactions_processed_noRNAcentralIDs.pkl")

Unique RNA entities: 29074


In [14]:
from collections import defaultdict

# Initialize counter
db_counts = defaultdict(int)

for buckets in rna_entities['ID_buckets']:
    for db, ids in buckets.items():
        if db != 'RNAcentral':  # exclude placeholder
            db_counts[db] += len(set(ids))  # count unique IDs per entity

# Print summary
for db, count in db_counts.items():
    print(f"{db}: {count} unique IDs across entities")


miRBase: 6447 unique IDs across entities
NCBI: 18747 unique IDs across entities
Ensembl: 3884 unique IDs across entities
circBase: 1 unique IDs across entities
tRFdb: 0 unique IDs across entities
GenBank: 3 unique IDs across entities
Other: 0 unique IDs across entities


In [15]:
'''
To-discuss: Assumption: 
all annotated isoforms (transcripts) of a gene are considered potential interactors 
when RNAInter reports a gene-level interaction.

We could think of some filters on the isoforms. For now we download them all, and keep track of isoforms-per-interaction (A list of sequences)
'''

'\nTo-discuss: Assumption: \nall annotated isoforms (transcripts) of a gene are considered potential interactors \nwhen RNAInter reports a gene-level interaction.\n\nWe could think of some filters on the isoforms. For now we download them all, and keep track of isoforms-per-interaction (A list of sequences)\n'

<h3>Ensembl IDs to Ensemble transcripts to RNAcentral IDs</h3>

In [16]:
# Construct DUCK-DB based on RNAcentral BEFORE this step!!
import sys
sys.path.append("rnacentral-etl")

from rnacentral import QueryRnaCentral

qrc = QueryRnaCentral(base_dir="rnacentral_data/current_release", db_path="rnacentral-etl/rnacentral.duckdb")

In [17]:
'''
Adapt the code below based on the instructions:
Extract all RNAcentral IDS of the ensemble transcripts related to ensemble GeneIDs that satisfy the conditions (Tax ID and rna-type)
Extract from row.get("ID_buckets", {}).get("Ensembl", [])
and place in RNAcentral a list for each transcript, so that we have a list of lists. row.get("ID_buckets", {}).get("RNAcentral", [])

We are not sure if all ENSEMBL IDs we have are gene ids, so first try with gene, if nothing try with transcript ID.
'''

# Usefoul code: 
taxid= "10090"
res = qrc.ensembl_gene_to_rnacentral_transcripts("ENSMUSG00000065862", taxid) #for ENS. gene ID
# qrc.ensembl_to_rnacentral('ENSMUST00000083928', taxid) # ENSMUSG00000065862 for ENS transcript ID
'''
Ensure 'rna_type' returned by query rna central is the same rna_type of the row.
First look if the rnacentral type is in the standard categories: Standardised categories: {'tRNA', 'miRNA', 'rRNA', 'pseudo', 'lncRNA', 'piRNA', 'tRF', 'mRNA', 'sRNA', 'lincRNA', 'circRNA', 'ncRNA', 'ribozyme', 'snoRNA', 'scaRNA', 'snRNA', 'scRNA', 'sncRNA'}
If not print it because we need to standardise everything.
'''

res

,RNAcentral_ID,rna_type,taxid,external_db
0,URS0000639553,snoRNA,10090,ENSEMBL


In [18]:
STANDARD_RNA_TYPES = {
    "tRNA", "miRNA", "rRNA", "pseudo", "lncRNA", "piRNA",
    "sRNA", "lincRNA", "ncRNA", "ribozyme", "snoRNA",
    "scaRNA", "snRNA", "scRNA", "sncRNA"
}
ACCEPTABLE_TYPE_MAP = {
    "SRP_RNA": ["ncRNA"],
    "RNase_MRP_RNA": ["ncRNA", "ribozyme"],
    "pre_miRNA": ["miRNA"],          # precursor -> mature
    "scaRNA": ["snoRNA", "ncRNA"],
    "snRNA": ["ncRNA"]               # small nuclear RNA -> generic ncRNA
}


def get_rnacentral_ids(row):
    """
    Populate row['ID_buckets']['RNAcentral'] with:
    a list of lists (one list per Ensembl ID).
    """
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    ensembl_ids = id_buckets.get("Ensembl", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")

    if not ensembl_ids or taxid is None:
        return row

    rnacentral_per_ensembl = []
    
    for ensembl_id in ensembl_ids:
        rnacentral_ids = set()

        try:
            # 1) try Ensembl gene → RNAcentral transcripts
            df = qrc.ensembl_gene_to_rnacentral_transcripts(
                ensembl_id, taxid
            )

            # 2) fallback: Ensembl transcript → RNAcentral
            if df is None or df.empty:
                df = qrc.ensembl_to_rnacentral(
                    ensembl_id, taxid
                )
                if df is not None and not df.empty:
                    print('Transcript detected')
                    print(df)

            if df is None or df.empty:
                continue

            for _, r in df.iterrows():
                rc_id = r.get("RNAcentral_ID")
                rc_type = r.get("rna_type")

                if pd.isna(rc_id) or pd.isna(rc_type):
                    print('rnacentral data not valid')
                    continue

                if rc_type not in STANDARD_RNA_TYPES:
                    print(f"[UNSTANDARDISED RNA TYPE] {rc_type}, row type in this case: {row_rna_type}")
                
                # Check if the rc_type is OK for the row type
                if rc_type == row_rna_type:
                    rnacentral_ids.add(rc_id)
                elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                    # defensible bypass
                    rnacentral_ids.add(rc_id)
                else:
                    # truly mismatched type — log for review
                    print(f"[NO TYPE MATCH] {rc_type}, row type in this case: {row_rna_type}")

        except Exception:
            continue

        if rnacentral_ids:
            rnacentral_per_ensembl.append(list(rnacentral_ids))

    id_buckets["RNAcentral"] = rnacentral_per_ensembl if rnacentral_per_ensembl else []
    row["ID_buckets"] = id_buckets

    return row


rna_entities_trial = rna_entities.apply(
    get_rnacentral_ids, axis=1
)


[NO TYPE MATCH] lncRNA, row type in this case: pseudo
[NO TYPE MATCH] lncRNA, row type in this case: pseudo
[NO TYPE MATCH] ncRNA, row type in this case: snRNA
[NO TYPE MATCH] ncRNA, row type in this case: snRNA
[NO TYPE MATCH] sRNA, row type in this case: miRNA
[NO TYPE MATCH] ncRNA, row type in this case: snRNA
[UNSTANDARDISED RNA TYPE] misc_RNA, row type in this case: snRNA
[NO TYPE MATCH] misc_RNA, row type in this case: snRNA


In [19]:
from collections import defaultdict
import numpy as np

def compute_stats(df, target_db="RNAcentral"):
    """
    Compute row-level and ID-level statistics for ID_buckets.
    Row = entity (primary unit).
    """

    # -------------------------
    # Row-level statistics
    # -------------------------
    db_rows_with_ids = defaultdict(int)
    db_ids_per_row = defaultdict(list)

    for _, row in df.iterrows():
        buckets = row.get("ID_buckets", {})
        for db, ids in buckets.items():
            if not ids:
                continue

            # count this row once per DB
            db_rows_with_ids[db] += 1

            # number of IDs in this row for this DB
            n_ids = 0
            for x in ids:
                n_ids += len(x) if isinstance(x, list) else 1

            db_ids_per_row[db].append(n_ids)

    print("=== Row-level (entity-level) statistics ===")
    for db, counts in db_ids_per_row.items():
        arr = np.array(counts)
        print(
            f"{db}: rows={db_rows_with_ids[db]} | "
            f"IDs/row: max={arr.max()}, mean={arr.mean():.2f}, median={np.median(arr)}"
        )

    # -------------------------
    # Row-level bins (IDs per row)
    # -------------------------
    print("\n=== IDs per row (binned) ===")
    for db, counts in db_ids_per_row.items():
        count_1 = np.sum(np.array(counts) == 1)
        count_2 = np.sum(np.array(counts) == 2)
        count_3_10 = np.sum((np.array(counts) >= 3) & (np.array(counts) <= 10))
        count_gt10 = np.sum(np.array(counts) > 10)

        print(
            f"{db}: 1={count_1}, 2={count_2}, "
            f"3–10={count_3_10}, >10={count_gt10}"
        )

    # -------------------------
    # ID-level statistics (secondary)
    # -------------------------
    db_unique_ids = defaultdict(set)

    for _, row in df.iterrows():
        buckets = row.get("ID_buckets", {})
        for db, ids in buckets.items():
            for x in ids:
                if isinstance(x, list):
                    db_unique_ids[db].update(x)
                else:
                    db_unique_ids[db].add(x)

    print("\n=== ID-level (secondary) statistics ===")
    for db, ids in db_unique_ids.items():
        print(f"{db}: {len(ids)} unique IDs")

    # -------------------------
    # Sequence availability (target DB)
    # -------------------------
    if target_db:
        has_seq = sum(
            bool(row.get("ID_buckets", {}).get(target_db))
            for _, row in df.iterrows()
        )
        no_seq = len(df) - has_seq

        print("\n=== Sequence availability ===")
        print(f"{target_db}: rows with IDs = {has_seq}")
        print(f"{target_db}: rows without IDs = {no_seq}")
        print(f"{target_db}: fraction retained = {has_seq / len(df):.3f}")


In [20]:
compute_stats(rna_entities_trial)

=== Row-level (entity-level) statistics ===
Ensembl: rows=3884 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=2134 | IDs/row: max=249, mean=4.88, median=1.0
NCBI: rows=18747 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6439 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3884, 2=0, 3–10=0, >10=0
RNAcentral: 1=1379, 2=183, 3–10=352, >10=220
NCBI: 1=18747, 2=0, 3–10=0, >10=0
miRBase: 1=6432, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3884 unique IDs
RNAcentral: 10378 unique IDs
NCBI: 18365 unique IDs
miRBase: 6317 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 2134
RNAcentral: rows without IDs = 26940
RNAcentral: fraction retained = 0.073


In [21]:
rna_entities_trial.to_pickle("rna_entities_trial.pkl")

In [39]:
# Construct DUCK-DB based on RNAcentral BEFORE this step!!
import sys
sys.path.append("rnacentral-etl")

from rnacentral import QueryRnaCentral

qrc = QueryRnaCentral(base_dir="rnacentral_data/current_release", db_path="rnacentral-etl/rnacentral.duckdb")

rna_entities_trial = pd.read_pickle("rna_entities_trial.pkl")

In [23]:
# -----------------------------
# Load NCBI → Ensembl mapping
# -----------------------------
ncbi_to_ensemble = pd.read_csv(
    "gene2ensembl",
    sep="\t",
    dtype=str,
    comment="#",
    header=None,
    low_memory=False,
)

ncbi_to_ensemble.columns = [
    "taxid",
    "geneid",
    "ensembl_gene",
    "refseq_rna_version",
    "ensembl_transcript_id",
    "refseq_protein_version",
    "ensembl_protein_id",
]

# enforce string typing
ncbi_to_ensemble["taxid"] = ncbi_to_ensemble["taxid"].astype(str)
ncbi_to_ensemble["geneid"] = ncbi_to_ensemble["geneid"].astype(str)

# collect all GeneIDs used in your dataset
all_ncbi_ids = set()

for buckets in rna_entities_trial["ID_buckets"]:
    all_ncbi_ids.update(map(str, buckets.get("NCBI", [])))

print(f"{len(all_ncbi_ids)} unique NCBI GeneIDs")

ncbi_to_ensemble_small = ncbi_to_ensemble[
    (ncbi_to_ensemble["ensembl_gene"] != "-") &
    (ncbi_to_ensemble["geneid"].isin(all_ncbi_ids))
]

ncbi2ens_gene = (
    ncbi_to_ensemble_small
    .groupby(["taxid", "geneid"])["ensembl_gene"]
    .unique()
    .to_dict()
)
print(len(ncbi2ens_gene))

import pickle
from functools import lru_cache

# key = rc_type from RNAcentral, value = list of row types it can match
ACCEPTABLE_TYPE_MAP = {
    "SRP_RNA": ["ncRNA"],
    "RNase_MRP_RNA": ["ncRNA", "ribozyme"],
    "pre_miRNA": ["miRNA"],          # precursor -> mature
    "scaRNA": ["snoRNA", "ncRNA"],
    "snRNA": ["ncRNA"]               # small nuclear RNA -> generic ncRNA
}

18365 unique NCBI GeneIDs
12463


<h3>NCBI IDs to refSeq IDs to RNAcentral IDs</h3>

In [40]:
from collections import defaultdict
from functools import lru_cache

gene2refseq_index = defaultdict(list)

with open("gene2refseq", "r") as f:   # <-- FIX: no gzip
    for line in f:
        if line.startswith("#"):
            continue
        cols = line.rstrip("\n").split("\t")

        taxid = cols[0]
        geneid = cols[1]
        rna_acc = cols[3]   # RNA_nucleotide_accession
        status = cols[2]

        if status == "SUPPRESSED":
            continue

        if rna_acc == "-":
            continue

        if rna_acc.startswith(("NR_", "XR_")):
            rna_type = "ncRNA"
        #elif rna_acc.startswith(("NM_", "XM_")):
        #    continue
            #rna_type = "mRNA"
        else:
            continue

        gene2refseq_index[(taxid, geneid)].append({
            "refseq_id": rna_acc,
            "rna_type": rna_type,
            "status": status
        })



In [41]:
@lru_cache(maxsize=100_000)
def ncbi_gene_to_refseq_transcripts(ncbi_gene_id, taxid):
    key = (str(ncbi_gene_id), str(taxid))  # cast to str
    return gene2refseq_index.get(key, [])

print(ncbi_gene_to_refseq_transcripts(2711, 102577949))

refseq_id = "NR_036832"  # example RefSeq
taxid = "58123"
print(f"\nRefSeq to RNAcentral (RefSeq={refseq_id}, taxid={taxid}):")
df_ref = qrc.refseq_to_rnacentral(refseq_id, taxid)
print(df_ref)

[{'refseq_id': 'XR_003065960.2', 'rna_type': 'ncRNA', 'status': 'MODEL'}, {'refseq_id': 'XR_371036.4', 'rna_type': 'ncRNA', 'status': 'MODEL'}]

RefSeq to RNAcentral (RefSeq=NR_036832, taxid=58123):
   RNAcentral_ID rna_type
0  URS0000000036     rRNA


In [42]:

errors_standard_types = []
errors_no_match_types = []

def get_rnacentral_from_ncbi_ids_refseq(row):
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    ncbi_ids = id_buckets.get("NCBI", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")

    if not ncbi_ids or not taxid:
        return row

    # do not overwrite existing RNAcentral mappings
    if id_buckets.get("RNAcentral"):
        return row

    rnacentral_per_ncbi = []

    for ncbi_id in ncbi_ids:
        rnacentral_ids = set()
        try:
            refseq_records = ncbi_gene_to_refseq_transcripts(
                taxid, ncbi_id
            )
            if not refseq_records:
                continue
            #print('taxid', taxid)
            #print('ncbi_id', ncbi_id)
            #print('refseq_records', refseq_records)

            for rec in refseq_records:
                refseq_id = rec.get("refseq_id")
                refseq_id = refseq_id.split(".")[0]
                status = rec.get("status")
                #print('refseq_id', refseq_id)

                if not refseq_id:
                    continue

                df_rnacentral = qrc.refseq_to_rnacentral(str(refseq_id), str(taxid))

                if df_rnacentral is None or df_rnacentral.empty:
                    continue
                
                # For loop for Rnacentral items:
                for _, row_rc in df_rnacentral.iterrows():
                    rnacentral_id = row_rc.get('RNAcentral_ID')
                    rc_type = row_rc.get('rna_type')
                    #print('rnacentral_id', rnacentral_id)
                    #print('rc_type', rc_type)
                    #print('row_rna_type', row_rna_type)

                    if rc_type not in STANDARD_RNA_TYPES:
                        errors_standard_types.append(
                            f"[UNSTANDARDISED RNA TYPE] {rc_type} (refseq_id:{refseq_id})"
                        )
                        
                    # Check if the rc_type is OK for the row type
                    if rc_type == row_rna_type:
                        rnacentral_ids.add(rnacentral_id)
                    elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                        # defensible bypass
                        rnacentral_ids.add(rnacentral_id)
                    else:
                        # truly mismatched type — log for review
                        errors_no_match_types.append(f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}")

        except Exception as e:
            print("Error:", e)
            continue

        if rnacentral_ids:
            rnacentral_per_ncbi.append(list(rnacentral_ids))

    id_buckets["RNAcentral"] = rnacentral_per_ncbi
    row["ID_buckets"] = id_buckets
    return row


rna_entities_trial_stepTwo = rna_entities_trial.progress_apply(
    get_rnacentral_from_ncbi_ids_refseq,
    axis=1
)

rna_entities_trial_stepTwo.to_pickle("rna_entities_trial_stepTwo.pkl")

compute_stats(rna_entities_trial_stepTwo)

Querying RNAcentral from NCBI IDs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 29074/29074 [20:49<00:00, 23.27it/s]


=== Row-level (entity-level) statistics ===
Ensembl: rows=3884 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=7307 | IDs/row: max=249, mean=2.51, median=1.0
NCBI: rows=18747 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6439 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3884, 2=0, 3–10=0, >10=0
RNAcentral: 1=5458, 2=763, 3–10=831, >10=255
NCBI: 1=18747, 2=0, 3–10=0, >10=0
miRBase: 1=6432, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3884 unique IDs
RNAcentral: 18250 unique IDs
NCBI: 18365 unique IDs
miRBase: 6317 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 7307
RNAcentral: rows without IDs = 21767
RNAcentral: fraction retained = 0.251


<h3>NCBI IDs to Ensembl IDs to Ensemble transcripts to RNAcentral IDs</h3>

In [43]:
@lru_cache(maxsize=100_000)
def cached_ensembl_gene_to_rnacentral(ensembl_gene, taxid):
    return qrc.ensembl_gene_to_rnacentral_transcripts(ensembl_gene, taxid)

# -----------------------------
# Main function
# -----------------------------
errors_standard_types = []
errors_no_match_types = []
def get_rnacentral_from_ncbi_ids(row):
    """
    Starting from NCBI GeneIDs:
    - map to Ensembl transcripts via gene2ensembl
    - query RNAcentral at transcript level
    - enforce RNA type consistency
    - store list-of-lists in ID_buckets['RNAcentral']
      (one list per NCBI GeneID)
    - append to existing RNAcentral IDs if present
    """
    row = row.copy()   # critical
    id_buckets = row.get("ID_buckets", {})
    ncbi_ids = id_buckets.get("NCBI", [])
    taxid = str(row.get("Species"))
    row_rna_type = row.get("Category")  

    if not ncbi_ids or taxid is None:
        return row

    # skip existing RNAcentral entries, if any
    existing = id_buckets.get("RNAcentral", [])
    if existing:
        #print(existing)
        #print(id_buckets)
        return row
    rnacentral_per_ncbi = []

    for ncbi_id in ncbi_ids:
        rnacentral_ids = set()

        try:
            # NCBI GeneID → Ensembl Gene IDs
            ens_genes = ncbi2ens_gene.get((taxid, str(ncbi_id)), [])
            
            for ensembl_gene in ens_genes:
                #df = qrc.ensembl_gene_to_rnacentral_transcripts(
                #    ensembl_gene, taxid
                #)
                df = cached_ensembl_gene_to_rnacentral(ensembl_gene, taxid)

                if df is None or df.empty:
                    continue

                for _, r in df.iterrows():
                    rc_id = r.get("RNAcentral_ID") or r.get("rnacentral_id")
                    rc_type = r.get("rna_type")
                    
                    if pd.isna(rc_id) or pd.isna(rc_type):
                        continue

                    # Normalize unstandardised RNA types if needed
                    if rc_type not in STANDARD_RNA_TYPES:
                        errors_standard_types.append(f"[UNSTANDARDISED RNA TYPE] {rc_type} for row type {row_rna_type}")
                    
                    # Check if the rc_type is OK for the row type
                    if rc_type == row_rna_type:
                        rnacentral_ids.add(rc_id)
                    elif rc_type in ACCEPTABLE_TYPE_MAP and row_rna_type in ACCEPTABLE_TYPE_MAP[rc_type]:
                        # defensible bypass
                        rnacentral_ids.add(rc_id)
                    else:
                        # truly mismatched type — log for review
                        errors_no_match_types.append(f"[no match RNA TYPE] {rc_type} for row type {row_rna_type}")
        except Exception:
            continue

        # Only append non-empty lists
        if rnacentral_ids:
            rnacentral_per_ncbi.append(list(rnacentral_ids))

    # If nothing found and no previous entries, make it empty
    id_buckets["RNAcentral"] = rnacentral_per_ncbi if rnacentral_per_ncbi else []
    row["ID_buckets"] = id_buckets

    return row

# -----------------------------
# Apply to dataframe
# -----------------------------

from tqdm import tqdm

tqdm.pandas(desc="Querying RNAcentral from NCBI IDs")

rna_entities_trial_stepThree = rna_entities_trial_stepTwo.progress_apply(
    get_rnacentral_from_ncbi_ids,
    axis=1
)
'''
# Save the counters to disk
with open("errors_standard_types.pkl", "wb") as f:
    pickle.dump(errors_standard_types, f)

with open("errors_no_match_types.pkl", "wb") as f:
    pickle.dump(errors_no_match_types, f)

# Load them back later if needed
with open("errors_standard_types.pkl", "rb") as f:
    errors_standard_types_loaded = pickle.load(f)

with open("errors_no_match_types.pkl", "rb") as f:
    errors_no_match_types_loaded = pickle.load(f)
'''
rna_entities_trial_stepThree.to_pickle("rna_entities_trial_stepThree.pkl")

compute_stats(rna_entities_trial_stepThree)

Querying RNAcentral from NCBI IDs: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 29074/29074 [10:06<00:00, 47.96it/s]


=== Row-level (entity-level) statistics ===
Ensembl: rows=3884 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=7669 | IDs/row: max=249, mean=2.78, median=1.0
NCBI: rows=18747 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6439 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3884, 2=0, 3–10=0, >10=0
RNAcentral: 1=5690, 2=778, 3–10=886, >10=315
NCBI: 1=18747, 2=0, 3–10=0, >10=0
miRBase: 1=6432, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3884 unique IDs
RNAcentral: 20711 unique IDs
NCBI: 18365 unique IDs
miRBase: 6317 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 7669
RNAcentral: rows without IDs = 21405
RNAcentral: fraction retained = 0.264


<h3>miRbase IDs to mature miRbase IDs to RNAcentral IDs (TODO)</h3>

In [44]:
#import pandas as pd
#rna_entities_trial_stepThree = pd.read_pickle("rna_entities_trial_stepThree.pkl")

In [45]:
qrc.mirbase_to_rnacentral('MIMAT0000434', '9606')

,RNAcentral_ID,rna_type,taxid
0,URS00002620A7,miRNA,9606


In [46]:
from Bio import SeqIO
from collections import defaultdict

# precursor_id -> list of mature IDs (MIMAT…)
precursor_to_mature_ids = defaultdict(list)

# mature_id -> sequence
mature_to_sequence = defaultdict(list)

# precursor_id -> fallback sequences
precursor_to_sequence = defaultdict(list)

for record in SeqIO.parse("miRNA.dat", "embl"):
    precursor_id = record.id  # MI000...

    for feature in record.features:
        if feature.type not in ("miRNA", "mature_miRNA"):
            continue

        seq = feature.location.extract(record).seq.upper().replace("T", "U")
        quals = feature.qualifiers

        mat_acc = quals.get("accession", [None])[0]

        if mat_acc:  # mature miRNA
            mature_to_sequence[mat_acc].append(str(seq))
            precursor_to_mature_ids[precursor_id].append(mat_acc)
        else:
            # precursor-only sequence (fallback)
            precursor_to_sequence[precursor_id].append(str(seq))


In [47]:
def get_mirna_sequences(row):
    row = row.copy()
    id_buckets = row.get("ID_buckets", {})
    taxid = str(row.get("Species"))

    mirbase_ids = id_buckets.get("miRBase", [])
    rnacentral_per_mirbase = []
    seqs = set()

    if not mirbase_ids:
        return row

    rnacentral_ids = set()

    for mid in mirbase_ids:
        mature_ids = []

        if mid.startswith("MIMAT"):
            mature_ids = [mid]
        else:
            # precursor → mature IDs
            mature_ids = precursor_to_mature_ids.get(mid, [])

            # fallback: no mature annotated
            if not mature_ids:
                seqs.update(precursor_to_sequence.get(mid, []))
                continue

        for mature_id in mature_ids:
            # sequences
            seqs.update(mature_to_sequence.get(mature_id, []))

            # RNAcentral mapping
            df_rc = qrc.mirbase_to_rnacentral(mature_id, taxid)
            if df_rc is not None and not df_rc.empty:
                rnacentral_ids.update(df_rc["RNAcentral_ID"].tolist())

    if rnacentral_ids:
        rnacentral_per_mirbase.append(list(rnacentral_ids))

    row["sequences"] = list(seqs)
    id_buckets["RNAcentral"] = rnacentral_per_mirbase
    row["ID_buckets"] = id_buckets

    return row


In [48]:
rna_entities_trial_stepFour = rna_entities_trial_stepThree.apply(
    get_mirna_sequences, axis=1
)

In [49]:
compute_stats(rna_entities_trial_stepFour)

=== Row-level (entity-level) statistics ===
Ensembl: rows=3884 | IDs/row: max=1, mean=1.00, median=1.0
RNAcentral: rows=14034 | IDs/row: max=249, mean=2.02, median=1.0
NCBI: rows=18747 | IDs/row: max=1, mean=1.00, median=1.0
miRBase: rows=6439 | IDs/row: max=3, mean=1.00, median=1.0
GenBank: rows=3 | IDs/row: max=1, mean=1.00, median=1.0
circBase: rows=1 | IDs/row: max=1, mean=1.00, median=1.0

=== IDs per row (binned) ===
Ensembl: 1=3884, 2=0, 3–10=0, >10=0
RNAcentral: 1=11386, 2=1442, 3–10=891, >10=315
NCBI: 1=18747, 2=0, 3–10=0, >10=0
miRBase: 1=6432, 2=6, 3–10=1, >10=0
GenBank: 1=3, 2=0, 3–10=0, >10=0
circBase: 1=1, 2=0, 3–10=0, >10=0

=== ID-level (secondary) statistics ===
Ensembl: 3884 unique IDs
RNAcentral: 25418 unique IDs
NCBI: 18365 unique IDs
miRBase: 6317 unique IDs
GenBank: 3 unique IDs
circBase: 1 unique IDs

=== Sequence availability ===
RNAcentral: rows with IDs = 14034
RNAcentral: rows without IDs = 15040
RNAcentral: fraction retained = 0.483


<h1>From here the code is ----work-in-progress----</h1>

<h3>Final step for sequence conversion: Symbol from RNAInter</h3>

In [ ]:
# we start from rna_entities_trial_stepFour
# 1. for each row we search if RNAcentral has a corresponding external ID, with related taxID and ncRNA type.
# 2. if the corresponding ID exists, and RNAcentral id bucket is empty, add the id/s to the bucket. Nice!
# 3. if the corresponding ID exist, and it is only one, and RNAcentral id bucket is not empty: substitute RNAcentral id bucket with the new RNAcentral ID
# 4. if the corresponding ID exist, and it is > 1, and RNAcentral id bucket is not empty: add new RNAcentral IDs, if not present already.

# Keep track of point 1, 2, 3 and 4 numbers (How many we added.)

In [60]:
rna_entities_trial_stepFour

,Category,ID_buckets,RNA_IDs,Raw_ID_key,Species,Symbol,sequences
0,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065862],"(Ensembl:ENSMUSG00000065862,)",10090,Gm24029,NaN
1,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065282],"(Ensembl:ENSMUSG00000065282,)",10090,Gm24455,NaN
2,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065208],"(Ensembl:ENSMUSG00000065208,)",10090,Gm24616,NaN
3,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000077575],"(Ensembl:ENSMUSG00000077575,)",10090,Gm24888,NaN
4,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065904],"(Ensembl:ENSMUSG00000065904,)",10090,Gm26109,NaN
...,...,...,...,...,...,...,...
33408,lncRNA,"{'miRBase': [], 'NCBI': ['101060385'], 'Ensemb...",[NCBI:101060385],"(NCBI:101060385,)",9606,LINC01939,NaN
33409,lncRNA,"{'miRBase': [], 'NCBI': ['101930370'], 'Ensemb...",[NCBI:101930370],"(NCBI:101930370,)",9606,LOC101930370,NaN
33410,lncRNA,"{'miRBase': [], 'NCBI': ['282566'], 'Ensembl':...",[NCBI:282566],"(NCBI:282566,)",9606,Linc00515,NaN
33411,miRNA,"{'miRBase': ['MIMAT0000425'], 'NCBI': [], 'Ens...",[miRBase:MIMAT0000425],"(miRBase:MIMAT0000425,)",9606,hsa-miR-130a-5p,[CAGUGCAAUGUUAAAAGGGCAU]


In [ ]:
compute_stats(rna_entities_trial_stepFive)

<h2>RNAcentral sequence retrieval, and GO terms retrieval</h2>

In [ ]:
# input dataframe: rna_entities_trial_stepFour
# 1. retrieve RNAcentral sequences: rnacentral_to_sequence -> list in 'sequences' column
# 2. retrieve GO terms and rfam_identifiers with rnacentral_metadata -> add 'GO' column, NaN if no GO, otherwhise list of correlated GO terms
# 3. RfamId, same as GO terms

'''
urs_id = "URS000000000F"  # example RNAcentral ID 
print(f"\nRNAcentral metadata for URS={urs_id}:")
df_meta = qrc.rnacentral_metadata(urs_id)
print(df_meta) # dataframe, possible multiple results, take unique rfam IDs and unique GO ids
# keys of this dataframe: go_id  go_rfam_id  rfam_id  rfam_description

urs_id = "URS00000478B7"
print(f"\nSequence for RNAcentral ID={urs_id}:")
seq_result = qrc.rnacentral_to_sequence(urs_id)
print(seq_result[0]) # this is the sequence.
# Sequence transformations todo: 1. all upper cases, 2. [A U C G], so T->U. 3. Remove any other character, if any.'''



In [57]:
from collections import defaultdict
import pandas as pd
import re
from tqdm import tqdm

# ----------------------------
# Helper functions
# ----------------------------

def clean_rna_sequence(seq: str) -> str:
    """
    Clean RNA sequence:
    - uppercase
    - convert T -> U
    - keep only A/U/C/G
    """
    seq = seq.upper().replace("T", "U")
    seq = re.sub(r"[^AUCG]", "", seq)
    return seq

def get_rnacentral_sequences(id_list):
    """
    Retrieve sequences from RNAcentral for a list of RNAcentral IDs
    Returns a list of sequences (one per ID)
    """
    sequences = []
    for rc_id in id_list:
        try:
            seq = qrc.rnacentral_to_sequence(rc_id)
            # take first sequence if multiple, clean it
            s = seq[0]
            sequences.append(clean_rna_sequence(s))
        except Exception:
            continue
    return sequences

def get_rnacentral_metadata(id_list):
    """
    Retrieve GO terms and Rfam IDs from RNAcentral for a list of RNAcentral IDs
    Returns two lists: go_terms, rfam_ids
    """
    go_terms = set()
    rfam_ids = set()
    for rc_id in id_list:
        try:
            df_meta = qrc.rnacentral_metadata(rc_id)
            if df_meta is not None and not df_meta.empty:
                go_terms.update(df_meta['go_id'].dropna().unique())
                rfam_ids.update(df_meta['rfam_id'].dropna().unique())
        except Exception:
            continue
    return list(go_terms) if go_terms else pd.NA, list(rfam_ids) if rfam_ids else pd.NA

# ----------------------------
# Apply to DataFrame
# ----------------------------

tqdm.pandas(desc="Fetching RNAcentral sequences & metadata")

def enrich_rnacentral(row):
    """
    row['ID_buckets']['RNAcentral'] is expected to be list of lists of RNAcentral IDs
    """
    rc_buckets = row.get("ID_buckets", {}).get("RNAcentral", [])
    # flatten list-of-lists
    rc_ids = [rc for sublist in rc_buckets for rc in sublist]

    # sequences
    row['sequences'] = get_rnacentral_sequences(rc_ids)

    # GO terms and Rfam IDs
    go_list, rfam_list = get_rnacentral_metadata(rc_ids)
    row['GO'] = go_list
    row['Rfam'] = rfam_list

    return row

rna_entities_final = rna_entities_trial_stepFour.progress_apply(
    enrich_rnacentral,
    axis=1
)


Fetching RNAcentral sequences & metadata: 100%|████████████████████████████████████████████████████████████████████████████████████████████| 29074/29074 [06:33<00:00, 73.81it/s]


In [58]:
import pandas as pd

# Count rows with at least one sequence
count_with_seq = rna_entities_final['sequences'].apply(lambda x: bool(x)).sum()

# Count rows with at least one GO term
count_with_go = rna_entities_final['GO'].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()

# Count rows with at least one Rfam ID
count_with_rfam = rna_entities_final['Rfam'].apply(lambda x: isinstance(x, list) and len(x) > 0).sum()

total_rows = len(rna_entities_final)

print(f"Total rows: {total_rows}")
print(f"Rows with sequences: {count_with_seq} ({count_with_seq/total_rows:.2%})")
print(f"Rows with GO terms: {count_with_go} ({count_with_go/total_rows:.2%})")
print(f"Rows with Rfam IDs: {count_with_rfam} ({count_with_rfam/total_rows:.2%})")


Total rows: 29074
Rows with sequences: 14034 (48.27%)
Rows with GO terms: 1298 (4.46%)
Rows with Rfam IDs: 1391 (4.78%)


In [59]:
rna_entities_final

,Category,ID_buckets,RNA_IDs,Raw_ID_key,Species,Symbol,sequences,GO,Rfam
0,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065862],"(Ensembl:ENSMUSG00000065862,)",10090,Gm24029,[GUCCAAGGUGAUUCCCUCUCCAAAGGGCUGACUGUGUGCCCUUUU...,"[GO:0005730, GO:0006396]",[RF00272]
1,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065282],"(Ensembl:ENSMUSG00000065282,)",10090,Gm24455,[UUUGAGGUCUAUCCCGACGGGGCUUCCCUAGUAGCCAGACGUCGU...,"[GO:0005730, GO:0006396]",[RF00425]
2,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065208],"(Ensembl:ENSMUSG00000065208,)",10090,Gm24616,[GUACCAAAAGCUAGAUUUGUGGGGAAGGUUUAUUAAAUAACCUUA...,"[GO:0005730, GO:0006396]",[RF00394]
3,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000077575],"(Ensembl:ENSMUSG00000077575,)",10090,Gm24888,[GCCGAGGCUAGAGUCACAUCCUGACAUGGUGCUUGUCCUGGUGUG...,"[GO:0005730, GO:0006396]",[RF00334]
4,snoRNA,"{'miRBase': [], 'NCBI': [], 'Ensembl': ['ENSMU...",[Ensembl:ENSMUSG00000065904],"(Ensembl:ENSMUSG00000065904,)",10090,Gm26109,[GUGCGAUGAUGCAUUUUAUUCAACACAUCAUUCUGAAAAUAGAUG...,"[GO:0005730, GO:0006396]",[RF00154]
...,...,...,...,...,...,...,...,...,...
33408,lncRNA,"{'miRBase': [], 'NCBI': ['101060385'], 'Ensemb...",[NCBI:101060385],"(NCBI:101060385,)",9606,LINC01939,[GCGUCACCCCCACCAGCAGGUGCAGAGGUGCCGUUGCAGCAGUUG...,NaN,NaN
33409,lncRNA,"{'miRBase': [], 'NCBI': ['101930370'], 'Ensemb...",[NCBI:101930370],"(NCBI:101930370,)",9606,LOC101930370,[AGGGGGACCGGCGGUCCGGGCGAGGCCUCGGCCGCGCAUCGCACC...,NaN,NaN
33410,lncRNA,"{'miRBase': [], 'NCBI': ['282566'], 'Ensembl':...",[NCBI:282566],"(NCBI:282566,)",9606,Linc00515,[CCUGCCCUUCCUUGCAGCUGUGGCUCAGACAGGUAGCAUGGGCUC...,NaN,NaN
33411,miRNA,"{'miRBase': ['MIMAT0000425'], 'NCBI': [], 'Ens...",[miRBase:MIMAT0000425],"(miRBase:MIMAT0000425,)",9606,hsa-miR-130a-5p,[CAGUGCAAUGUUAAAAGGGCAU],NaN,NaN


<h2>FInalization: Joining Sequence Information with Interaction datset</h2>
<h6>We keep only interaction with avaiable sequences in both Interacting ncRNAs</h6>

In [ ]:
import pandas as pd

# ----------------------------
# Step 0: define a function to create a composite key
# ----------------------------
def make_entity_key(symbol, category, species, raw_id_list):
    """
    Create a tuple key combining Symbol, Category, Species, and sorted Raw_ID list
    """
    id_tuple = tuple(sorted(str(i) for i in raw_id_list if i is not None))
    return (symbol, category, species, id_tuple)

# ----------------------------
# Step 1: remove RNA entities without sequences
# ----------------------------
rna_entities_final = rna_entities_final.copy()
rna_entities_final = rna_entities_final[rna_entities_final['sequences'].apply(lambda x: bool(x))]
rna_entities_final.reset_index(drop=True, inplace=True)

# ----------------------------
# Step 2: add composite key
# ----------------------------
rna_entities_final['entity_key'] = rna_entities_final.apply(
    lambda row: make_entity_key(row['Symbol'], row['Category'], row['Species'], row['Raw_ID_key']),
    axis=1
)

# For interactions df, create keys for both interactors
df_filt_merged = df_filt.copy()

df_filt_merged['key1'] = df_filt_merged.apply(
    lambda row: make_entity_key(row['Interactor1.Symbol'], row['Category1'], row['Species1_TaxID'], row['Raw_ID1_list']),
    axis=1
)
df_filt_merged['key2'] = df_filt_merged.apply(
    lambda row: make_entity_key(row['Interactor2.Symbol'], row['Category2'], row['Species2_TaxID'], row['Raw_ID2_list']),
    axis=1
)

# ----------------------------
# Step 3: prepare mapping for merging
# ----------------------------
rna_map = rna_entities_final.set_index('entity_key')[
    ['sequences','GO','Rfam','ID_buckets']
]

# ----------------------------
# Step 4: merge sequences & metadata
# ----------------------------
# Interactor1
df_filt_merged = df_filt_merged.join(
    rna_map,
    on='key1',
    rsuffix='_1'
).rename(columns={
    'sequences_1': 'seq1_list',
    'GO_1': 'GO1',
    'Rfam_1': 'Rfam1',
    'ID_buckets_1': 'ID_buckets1'
})

# Interactor2
df_filt_merged = df_filt_merged.join(
    rna_map,
    on='key2',
    rsuffix='_2'
).rename(columns={
    'sequences_2': 'seq2_list',
    'GO_2': 'GO2',
    'Rfam_2': 'Rfam2',
    'ID_buckets_2': 'ID_buckets2'
})

# ----------------------------
# Step 5: drop interactions without sequences
# ----------------------------
df_filt_final = df_filt_merged[
    df_filt_merged['seq1_list'].apply(lambda x: bool(x)) &
    df_filt_merged['seq2_list'].apply(lambda x: bool(x))
].reset_index(drop=True)

print(f"Total interactions after filtering: {len(df_filt_final)}")

# ----------------------------
# Step 6: save final interaction dataframe
# ----------------------------
df_filt_final.to_pickle("interactions_with_sequences.pkl")


In [ ]:
# SAVE THE DATA
TODO

<h1>Statistics and graphics (TO-DO)</h1>

In [ ]:
# MAYBE IN ANOTHER FILE; LOAD DATA AND PERFORM STATISTICS